# Optional  Lab: Cost Function 


## Goals
In this lab you will:
- you will implement and explore the `cost` function for linear regression with one variable. 


## Tools
This self-contained version uses NumPy for arrays and Matplotlib for plots.
The plotting code is defined below, so no course helper module, custom style file,
or interactive Matplotlib backend is required.

Run the cells in order using this folder's `.venv` kernel. The plots are static by
default: change the parameters in a plotting cell and rerun it to explore.
See `README.md` for implementation notes and an optional slider example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Built-in settings only; no course style file or widget backend is needed.
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25})

## Problem Statement

You would like a model which can predict housing prices given the size of the house.  
Let's use the same two data points as before the previous lab- a house with 1000 square feet sold for \\$300,000 and a house with 2000 square feet sold for \\$500,000.


| Size (1000 sqft)     | Price (1000s of dollars) |
| -------------------| ------------------------ |
| 1                 | 300                      |
| 2                  | 500                      |


In [ ]:
x_train = np.array([1.0, 2.0])           #(size in 1000 square feet)
y_train = np.array([300.0, 500.0])           #(price in 1000s of dollars)

## Computing Cost
The term 'cost' in this assignment might be a little confusing since the data is housing cost. Here, cost is a measure how well our model is predicting the target price of the house. The term 'price' is used for housing data.

The equation for cost with one variable is:
  $$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2 \tag{1}$$ 
 
where 
  $$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{2}$$
  
- $f_{w,b}(x^{(i)})$ is our prediction for example $i$ using parameters $w,b$.  
- $(f_{w,b}(x^{(i)}) -y^{(i)})^2$ is the squared difference between the target value and the prediction.   
- These differences are summed over all the $m$ examples and divided by `2m` to produce the cost, $J(w,b)$.  
>Note, in lecture summation ranges are typically from 1 to m, while code will be from 0 to m-1.


The code below calculates cost by looping over each example. In each loop:
- `f_wb`, a prediction is calculated
- the difference between the target and the prediction is calculated and squared.
- this is added to the total cost.

In [ ]:
def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
      x (ndarray (m,)): Data, m examples 
      y (ndarray (m,)): target values
      w,b (scalar)    : model parameters  
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    cost_sum = 0 
    for i in range(m): 
        f_wb = w * x[i] + b   
        cost = (f_wb - y[i]) ** 2  
        cost_sum = cost_sum + cost  
    total_cost = (1 / (2 * m)) * cost_sum  

    return total_cost

## Cost Function Intuition

Your goal is to find a model $f_{w,b}(x) = wx + b$, with parameters $w,b$,  which will accurately predict house values given an input $x$. The cost is a measure of how accurate the model is on the training data.

The cost equation (1) above shows that if $w$ and $b$ can be selected such that the predictions $f_{w,b}(x)$ match the target data $y$, the $(f_{w,b}(x^{(i)}) - y^{(i)})^2 $ term will be zero and the cost minimized. In this simple two point example, you can achieve this!

In the previous lab, you determined that $b=100$ provided an optimal solution so let's set $b$ to 100 and focus on $w$.

<br/>
Below, change `w` and rerun the cell to find the value that minimizes cost. Try `w = 100`, `200`, and `300` while holding `b = 100`. The left plot shows residuals; the right plot shows the selected point on the cost curve.

In [ ]:
def plot_cost_intuition(x, y, w=200.0, b=100.0):
    """Show the model and its cost, with b held fixed along the cost curve."""
    w_values = np.linspace(0, 400, 201)
    costs = [compute_cost(x, y, candidate_w, b) for candidate_w in w_values]
    predictions = w * x + b
    selected_cost = compute_cost(x, y, w, b)
    x_line = np.linspace(x.min() - 0.2, x.max() + 0.2, 100)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    axes[0].scatter(x, y, marker="x", color="black", label="Training data")
    axes[0].plot(x_line, w * x_line + b, label=f"Prediction: {w:g}x + {b:g}")
    axes[0].vlines(x, y, predictions, colors="tab:red", linestyles="--",
                   label="Residuals (prediction - target)")
    axes[0].set(xlabel="Size (1000 sqft)", ylabel="Price (1000 dollars)",
                title="Linear model")
    axes[0].legend()

    axes[1].plot(w_values, costs, label=f"Cost with b = {b:g}")
    axes[1].scatter([w], [selected_cost], color="tab:red", zorder=3,
                    label=f"Selected w = {w:g}")
    axes[1].set(xlabel="w", ylabel="J(w, b)", title=f"Selected cost = {selected_cost:,.2f}")
    axes[1].legend()
    plt.show()


# Change w (try 100, 200, 300) and rerun this cell. Hold b fixed first.
w = 200.0
b = 100.0
plot_cost_intuition(x_train, y_train, w=w, b=b)

The plot contains a few points that are worth mentioning.
- cost is minimized when $w = 200$, which matches results from the previous lab
- Because the difference between the target and pediction is squared in the cost equation, the cost increases rapidly when $w$ is either too large or too small.
- Using the `w` and `b` selected by minimizing cost results in a line which is a perfect fit to the data.

## Cost Function Visualization - 3D

We can see how cost varies with both `w` and `b` using a contour plot and a 3D surface.
The code below creates a parameter grid with `np.meshgrid`, evaluates
`J(w, b) = mean((w*x + b - y)**2) / 2` on that grid, and plots the result directly.
This is the same cost as `compute_cost`; the grid calculation uses NumPy broadcasting
to evaluate many candidate parameter pairs at once.

### Larger Data Set
It is instructive to view a scenario with a few more data points. This data set includes data points that do not fall on the same line. What does that mean for the cost equation? Can we find $w$, and $b$ that will give us a cost of 0? 

In [ ]:
x_train = np.array([1.0, 1.7, 2.0, 2.5, 3.0, 3.2])
y_train = np.array([250, 300, 480,  430,   630, 730,])

Change `w_selected` and `b_selected` in the next cell, then rerun it.
The selected parameters are shown on the model fit, cost contours, and 3D surface.
Try to find a pair near the center of the lowest contours. No click handler is required.

In [ ]:
# Change these parameters and rerun this cell to explore the same cost surface.
w_selected = 209.0
b_selected = 2.4

w_values = np.linspace(-100, 500, 151)
b_values = np.linspace(-500, 500, 151)
W, B = np.meshgrid(w_values, b_values)
# W and B: (151, 151). Adding a final axis makes one prediction per sample.
# errors: (151, 151, m). Average over the sample axis to obtain J(w, b).
errors = W[..., np.newaxis] * x_train + B[..., np.newaxis] - y_train
J_grid = np.mean(errors ** 2, axis=-1) / 2
selected_cost = compute_cost(x_train, y_train, w_selected, b_selected)

fig = plt.figure(figsize=(16, 4.5), constrained_layout=True)
ax_fit = fig.add_subplot(1, 3, 1)
ax_contour = fig.add_subplot(1, 3, 2)
ax_surface = fig.add_subplot(1, 3, 3, projection="3d")

x_line = np.linspace(x_train.min() - 0.2, x_train.max() + 0.2, 100)
ax_fit.scatter(x_train, y_train, marker="x", color="black", label="Training data")
ax_fit.plot(x_line, w_selected * x_line + b_selected, label="Selected model")
ax_fit.vlines(x_train, y_train, w_selected * x_train + b_selected,
              colors="tab:red", linestyles="--", label="Residuals")
ax_fit.set(xlabel="Size (1000 sqft)", ylabel="Price (1000 dollars)",
           title=f"J = {selected_cost:,.2f}")
ax_fit.legend()

levels = np.geomspace(max(J_grid.min(), 1), J_grid.max(), 18)
contours = ax_contour.contour(W, B, J_grid, levels=levels, cmap="viridis")
ax_contour.clabel(contours, inline=True, fontsize=7, fmt="%.0f")
ax_contour.scatter([w_selected], [b_selected], color="tab:red", marker="x",
                   s=70, zorder=3, label="Selected (w, b)")
ax_contour.set(xlabel="w", ylabel="b", title="Cost contours")
ax_contour.legend()

ax_surface.computed_zorder = False
ax_surface.plot_surface(W, B, J_grid, cmap="viridis", alpha=0.8, zorder=1)
ax_surface.scatter([w_selected], [b_selected], [selected_cost],
                    color="tab:red", s=45, depthshade=False, zorder=5)
ax_surface.set(xlabel="w", ylabel="b", zlabel="J(w, b)", title="Cost surface")
ax_surface.ticklabel_format(axis="z", style="sci", scilimits=(0, 0))
ax_surface.view_init(elev=25, azim=-60)
plt.show()

Above, note the dashed lines in the left plot. These represent the portion of the cost contributed by each example in your training set. In this case, values of approximately $w=209$ and $b=2.4$ provide low cost. Note that, because our training examples are not on a line, the minimum cost is not zero.

### Convex Cost Surface

For this linear regression data, the cost is a convex quadratic in `w` and `b`.
The previous plot shows the actual housing-data cost surface. Its axes have different
scales, which can make the bowl shape harder to see.

The next plot is a separate, symmetric illustration, $z = u^2 + v^2$.
It makes the bowl shape easy to see, but is not the housing-data cost function.

In [ ]:
# A symmetric quadratic example, not the housing data's actual cost surface.
u_values = np.linspace(-2, 2, 81)
v_values = np.linspace(-2, 2, 81)
U, V = np.meshgrid(u_values, v_values)
Z = U ** 2 + V ** 2

fig = plt.figure(figsize=(7, 5), constrained_layout=True)
ax = fig.add_subplot(1, 1, 1, projection="3d")
ax.plot_surface(U, V, Z, cmap="viridis", alpha=0.85)
ax.set(xlabel="u", ylabel="v", zlabel="u squared + v squared",
        title="A symmetric convex bowl (illustration)")
plt.show()

# Congratulations!
You have learned the following:
 - The cost equation provides a measure of how well your predictions match your training data.
 - Minimizing the cost can provide optimal values of $w$, $b$.